In [92]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import os


# ----------------------------
# Training Process
# ----------------------------
def train_gan(num_epochs, noise_dim, target_dataset, noise_dataframe, feature_columns, weight_column, batch_size):
    # Load target dataset
    target_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=True)
    
    # Extract noise data and weights from dataframe
    noise_data = noise_dataframe[feature_columns].values
    noise_weights = noise_dataframe[weight_column].values
    noise_probs = noise_weights / noise_weights.sum()  # Normalize weights

    # Initialize models
    generator = Generator(input_dim=noise_dim, output_dim=target_dataset.data.shape[1]).to(device)
    discriminator = Discriminator(input_dim=target_dataset.data.shape[1]).to(device)

    # Optimizers
    optimizer_G = optim.Adam(generator.parameters(), lr=0.0002)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002)

    # Loss tracker
    G_losses = []
    D_losses = []

    for epoch in range(num_epochs):
        for real_samples, real_weights in target_loader:
            real_samples, real_weights = real_samples.to(device), real_weights.to(device)
            batch_size = real_samples.size(0)

            # ---------------------
            # Train Discriminator
            # ---------------------
            optimizer_D.zero_grad()

            # Real samples
            real_labels = torch.ones(batch_size, 1, device=device)
            real_predictions = discriminator(real_samples)
            real_loss = weighted_bce_loss(real_predictions, real_labels, real_weights)

            # Fake samples
            noise_indices = np.random.choice(len(noise_data), size=batch_size, p=noise_probs)
            noise = torch.tensor(noise_data[noise_indices], device=device).float()
            fake_samples = generator(noise)
            fake_labels = torch.zeros(batch_size, 1, device=device)
            fake_predictions = discriminator(fake_samples.detach())
            fake_loss = nn.BCELoss()(fake_predictions, fake_labels)

            # Total Discriminator loss
            d_loss = real_loss + fake_loss
            d_loss.backward()
            optimizer_D.step()

            # ---------------------
            # Train Generator
            # ---------------------
            optimizer_G.zero_grad()

            # Generate fake samples and classify as real
            fake_predictions = discriminator(fake_samples)
            g_loss = nn.BCELoss()(fake_predictions, real_labels)

            g_loss.backward()
            optimizer_G.step()

        # Log progress
        G_losses.append(g_loss.item())
        D_losses.append(d_loss.item())
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{num_epochs} | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

    return generator, G_losses, D_losses

In [86]:
drop_kwds = ['Gen', 'Weight_values', 'OS', 'group', 'gen', 'dataset', 'label', 'btag']

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

def drop_likes(df: 'pd.DataFrame', drop_kwd: 'list[str]' = []):
    """Drop columns containing the keywords in `drop_kwd`."""
    for kwd in drop_kwd:
        df = df.drop(columns=df.filter(like=kwd).columns, inplace=False)
    return df
    
def drop_and_scale(df):
    df = drop_likes(df, drop_kwds)
    df = df[df.weight>0]
    features = list(df.columns.values)
    df[features] = scaler.fit_transform(df[features])
    df["weight"] /= df["weight"].sum()
    return df

In [39]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pjoin = os.path.join
training_dir = '/Users/yuntongzhou/Desktop/Dihiggszztt/output/training'
dfD = pd.read_csv(pjoin(training_dir,'RegionD.csv'), index_col=0)
dfB = pd.read_csv(pjoin(training_dir,'RegionB.csv'), index_col=0)
dfA = pd.read_csv(pjoin(training_dir, "RegionA.csv"), index_col=0)
dfC = pd.read_csv(pjoin(training_dir, "RegionC.csv"), index_col=0)

In [ ]:
# Train GAN
num_epochs = 200
noise_dim = len(noise_features)
batch_size = int(len(noise_df)/2)
generator, G_losses, D_losses = train_gan(num_epochs, noise_dim, target_dataset, noise_df, noise_features, noise_weight, batch_size)

# Save generator output samples
noise_indices = np.random.choice(len(noise_df), size=16, p=noise_df[noise_weight].values)
test_noise = torch.tensor(noise_df.iloc[noise_indices][noise_features].values, device=device)
generated_samples = generator(test_noise).cpu().detach().numpy()
descaled_samples = scaler.inverse_transform(generated_samples)

/var/folders/w4/tl7q52611yg4x42kkpb7bm0m0000gn/T/ipykernel_99801/2176667534.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(data, dtype=torch.float), torch.tensor(weight, dtype=torch.float)


Epoch 0/200 | D Loss: 0.6695 | G Loss: 0.7190


In [ ]:
print("Generated Samples:", descaled_samples)